# 🎯 Garcia & Inoue approach

**MAC Address Format:** Handles colons (FD:07:0E:D5:28:AE)

**Columns:** timestamp=1, mac=3, rssi=4

In [1]:
# If running on Colab, these are usually present; keep to avoid missing-package errors.
!pip -q install pandas numpy scikit-learn scipy

import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from scipy.stats import entropy
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

print("pandas", pd.__version__)
print("numpy", np.__version__)


pandas 2.2.2
numpy 2.0.2


In [2]:
# If not using Colab, comment this cell out.
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
# ===== EDIT THESE TWO PATHS =====
LABEL_CSV = Path("/content/drive/MyDrive/Contest/ABC/Dataset/5f_label_loc_train.csv")
BLE_DIR   = Path("/content/drive/MyDrive/Contest/ABC/Dataset/BLE Data")

assert LABEL_CSV.exists(), f"Label CSV not found: {LABEL_CSV}"
assert BLE_DIR.exists(), f"BLE directory not found: {BLE_DIR}"
print("OK paths")


OK paths


In [4]:
def to_utc_naive(ts: pd.Series) -> pd.Series:
    # Input strings like "2023-04-10 14:21:46+09:00" or "2023-04-10T10:22:55.589+0900" are expected [file:5][file:6]
    out = pd.to_datetime(ts, errors="coerce", utc=True)
    return out.dt.tz_localize(None)

def normalize_room_label(x: str) -> str:
    if pd.isna(x):
        return x
    x = str(x).strip()
    return x.lower()  # keep consistent, since you have "nurse station", "cafeteria", room numbers, etc. [file:5]


Cell 5

In [5]:
df_label_raw = pd.read_csv(LABEL_CSV)
# Your label file includes columns like: activity, started_at, finished_at, user_id, room, floor [file:5]
required = {"activity", "started_at", "finished_at", "user_id", "room", "floor"}
missing = required - set(df_label_raw.columns)
assert not missing, f"Label CSV missing columns: {missing}. Found: {list(df_label_raw.columns)}"

df_label = df_label_raw.copy()
df_label["activity"] = df_label["activity"].astype(str).str.strip().str.lower()
df_label["floor"] = df_label["floor"].astype(str).str.strip().str.lower()
df_label["room"] = df_label["room"].map(normalize_room_label)

# Keep only 5th-floor location labels (labeler user_id==97 in your notes) [file:5]
df_label = df_label[df_label["activity"].eq("location")].copy()
df_label = df_label[df_label["user_id"].astype(str).eq("97")].copy()
df_label = df_label[df_label["floor"].str.contains("5th", na=False)].copy()

df_label["started_at"] = to_utc_naive(df_label["started_at"])
df_label["finished_at"] = to_utc_naive(df_label["finished_at"])
df_label = df_label.dropna(subset=["started_at", "finished_at", "room"]).copy()
df_label = df_label.sort_values("started_at").reset_index(drop=True)

assert len(df_label) > 0, "No 5th-floor location labels found after filtering."
print("Labels:", df_label.shape)
print("Label time span:", df_label["started_at"].min(), "->", df_label["finished_at"].max())
print("Rooms:", df_label["room"].nunique())
df_label.head(3)


Labels: (459, 11)
Label time span: 2023-04-10 05:21:46 -> 2023-04-13 08:18:25
Rooms: 23


,Unnamed: 0.1,Unnamed: 0,activity,started_at,finished_at,deleted_at,updated_at,user_id,user,room,floor
0,170,170,location,2023-04-10 05:21:46,2023-04-10 05:21:50,NaN,2023-04-10 05:22:02 UTC,97,5th-location,kitchen,5th
1,171,171,location,2023-04-10 05:21:55,2023-04-10 05:22:16,NaN,2023-04-10 05:23:02 UTC,97,5th-location,cafeteria,5th
2,172,172,location,2023-04-10 05:22:23,2023-04-10 05:23:25,NaN,2023-04-10 05:24:01 UTC,97,5th-location,kitchen,5th


Cell 6 — Beacon list

In [6]:
# Your BLE analysis notebook defines 25 MAC addresses for beacons [file:6]
MAC_LIST = [
    "F7:7F:78:76:7E:F3","C6:CD:5E:3D:2F:BB","D6:F4:3A:79:74:63","C9:17:55:E2:3E:0E","CA:60:AB:EE:EC:7F",
    "D6:51:7F:AB:0E:29","CC:54:33:F6:A7:90","EB:20:56:87:04:5A","EE:E7:46:DC:19:6F","C8:5B:BF:37:07:A0",
    "D7:26:F6:A3:44:D2","DD:83:B0:27:FD:36","E5:CD:4A:36:87:06","DC:22:B8:17:4E:B5","EA:09:20:80:D6:44",
    "E6:99:D1:EC:C6:81","F6:DA:97:C7:D5:28","EA:66:A1:12:2C:F4","C9:EA:57:8B:0F:80","D6:7C:1D:2C:2A:0A",
    "DA:E1:70:5F:44:97","DD:10:10:F6:4F:27","E6:F3:93:A8:9E:22","E6:60:05:1F:88:F9","D4:33:FD:F4:C2:A8",
]
MAC_LIST = [m.upper() for m in MAC_LIST]
BEACON_ID = {mac: i+1 for i, mac in enumerate(MAC_LIST)}  # 1..25
BEACON_COLS = [f"b{i:02d}" for i in range(1, 26)]

assert len(MAC_LIST) == 25
print("Beacons:", len(MAC_LIST), "=> columns:", BEACON_COLS[:5], "...")


Beacons: 25 => columns: ['b01', 'b02', 'b03', 'b04', 'b05'] ...


Cell 7 — Windowing + labeling (40s non-overlapping)




In [7]:
WINDOW_SECONDS = 40

def list_ble_files(ble_dir: Path):
    files = sorted([p for p in ble_dir.glob("*.csv")])
    assert len(files) > 0, f"No CSV files found in {ble_dir}"
    return files

def read_ble_csv(path: Path) -> pd.DataFrame:
    # Your BLE files are read with columns: user_id, timestamp, name, mac address, RSSI, power [file:6]
    df = pd.read_csv(
        path,
        header=None,
        names=["user_id", "timestamp", "name", "mac", "rssi", "power"],
        usecols=[0, 1, 2, 3, 4, 5],
        low_memory=False
    )
    return df

def assign_room_to_windows(df_windows: pd.DataFrame, df_intervals: pd.DataFrame) -> pd.Series:
    # merge_asof on started_at, then validate winmid <= finished_at
    w = df_windows[["winmid"]].sort_values("winmid").reset_index()
    inter = df_intervals[["started_at", "finished_at", "room"]].sort_values("started_at").reset_index(drop=True)

    merged = pd.merge_asof(
        w, inter,
        left_on="winmid",
        right_on="started_at",
        direction="backward",
        allow_exact_matches=True
    )

    inside = (merged["winmid"] >= merged["started_at"]) & (merged["winmid"] <= merged["finished_at"])
    merged.loc[~inside, "room"] = np.nan

    out = merged.set_index("index")["room"].reindex(df_windows.index)
    return out

def build_windows_from_ble_files(files, df_intervals):
    rows = []
    for k, p in enumerate(files, 1):
        df = read_ble_csv(p)
        if df.empty:
            continue

        # caregiver BLE collector is user_id==90 in your notes [file:5][file:6]
        df = df[df["user_id"].astype(str).eq("90")].copy()
        if df.empty:
            continue

        df["mac"] = df["mac"].astype(str).str.upper().str.strip()
        df = df[df["mac"].isin(MAC_LIST)].copy()
        if df.empty:
            continue

        df["ts"] = to_utc_naive(df["timestamp"])
        df["rssi"] = pd.to_numeric(df["rssi"], errors="coerce")
        df = df.dropna(subset=["ts", "rssi"]).copy()
        if df.empty:
            continue

        # Non-overlapping 45s windows
        df["winstart"] = df["ts"].dt.floor(f"{WINDOW_SECONDS}S")
        df["winmid"] = df["winstart"] + pd.to_timedelta(WINDOW_SECONDS/2, unit="s")

        # per-window global RSSI stats (5 statistical features)
        g = df.groupby("winstart")["rssi"]
        win_stats = g.agg(["mean", "std", "min", "max", "count"]).reset_index()
        win_stats = win_stats.rename(columns={
            "mean": "rssi_mean",
            "std": "rssi_std",
            "min": "rssi_min",
            "max": "rssi_max",
            "count": "rssi_count"
        })
        win_stats["rssi_std"] = win_stats["rssi_std"].fillna(0.0)

        # pivot beacon means (25 fixed columns), fill missing with 0 (paper uses 0 for missing) [file:3]
        df["beacon_id"] = df["mac"].map(BEACON_ID).astype(int)
        pvt = df.groupby(["winstart", "beacon_id"])["rssi"].mean().unstack("beacon_id")
        pvt = pvt.reindex(columns=range(1, 26)).fillna(0.0)
        pvt.columns = BEACON_COLS
        pvt = pvt.reset_index()

        out = win_stats.merge(pvt, on="winstart", how="left")
        out["winmid"] = out["winstart"] + pd.to_timedelta(WINDOW_SECONDS/2, unit="s")

        # 3 temporal features
        out["hour"] = out["winmid"].dt.hour.astype(int)
        out["minute"] = out["winmid"].dt.minute.astype(int)
        out["microsecond"] = out["winmid"].dt.microsecond.astype(int)

        # Attach room labels
        out["room"] = assign_room_to_windows(out, df_intervals)

        rows.append(out)

        if k % 250 == 0:
            print(f"Processed {k}/{len(files)} files; windows so far: {sum(len(x) for x in rows)}")

    if not rows:
        raise RuntimeError("No BLE windows created. Check BLE files, user_id filter, or MAC filtering.")
    dfw = pd.concat(rows, ignore_index=True)
    return dfw

ble_files = list_ble_files(BLE_DIR)
print("BLE files:", len(ble_files))

# Add to Cell 7 - after building windows
def apply_six_beacon_filtering(df_windows, room_to_six):
    """Keep only 6 strongest beacons per room, zero out rest"""
    df_filtered = df_windows.copy()

    for room, six_cols in room_to_six.items():
        mask = df_filtered["room"] == room
        non_six = [c for c in BEACON_COLS if c not in six_cols]
        df_filtered.loc[mask, non_six] = 0.0

    return df_filtered

df_windows = build_windows_from_ble_files(ble_files, df_label)
print("Windows:", df_windows.shape)
print("Labeled windows:", df_windows["room"].notna().sum())
df_windows.head(3)


BLE files: 4107
Processed 500/4107 files; windows so far: 696
Processed 1000/4107 files; windows so far: 1325
Processed 2000/4107 files; windows so far: 1706
Processed 2250/4107 files; windows so far: 2064
Processed 2500/4107 files; windows so far: 2390
Processed 3000/4107 files; windows so far: 2684
Processed 3250/4107 files; windows so far: 3021
Processed 3750/4107 files; windows so far: 3736
Processed 4000/4107 files; windows so far: 4089
Windows: (4250, 36)
Labeled windows: 2331


,winstart,rssi_mean,rssi_std,rssi_min,rssi_max,rssi_count,b01,b02,b03,b04,...,b21,b22,b23,b24,b25,winmid,hour,minute,microsecond,room
0,2023-04-10 01:22:40,-94.760000,2.429351,-98,-89,150,0.0,0.0,0.000000,-98.000000,...,0.0,0.0,0.0,0.0,0.0,2023-04-10 01:23:00,1,23,0,NaN
1,2023-04-10 01:22:40,-92.839422,2.976474,-100,-86,1177,0.0,0.0,-96.333333,-93.551724,...,0.0,0.0,0.0,0.0,0.0,2023-04-10 01:23:00,1,23,0,NaN
2,2023-04-10 01:22:40,-92.839286,2.364716,-96,-88,56,0.0,0.0,0.000000,-92.666667,...,0.0,0.0,0.0,0.0,0.0,2023-04-10 01:23:00,1,23,0,NaN


Cell 8 — Date split (3.15 days total) + encode labels

In [9]:
# ========================================
# Cell 8 — Stratified Split (Keep ALL 22 Classes)
# ========================================

from sklearn.model_selection import train_test_split  # ← ADD THIS

df_windows = df_windows.sort_values("winmid").reset_index(drop=True)

df_labeled = df_windows[df_windows["room"].notna()].copy()
assert len(df_labeled) > 0, "No labeled windows"

# Build encoding WITHOUT filtering
rooms = sorted(df_labeled["room"].unique().tolist())
room2id = {r: i for i, r in enumerate(rooms)}
id2room = {i: r for r, i in room2id.items()}
df_labeled["y"] = df_labeled["room"].map(room2id).astype(int)

print(f"Total labeled: {len(df_labeled)}")
print(f"Classes: {len(rooms)}")
print(f"Time: {df_labeled['winmid'].min()} -> {df_labeled['winmid'].max()}")

# Check which classes can't be stratified
counts = df_labeled["room"].value_counts()
single_sample = counts[counts == 1]
two_sample = counts[counts == 2]
multi_sample = counts[counts >= 2]

print(f"\nClasses with 1 sample: {len(single_sample)}")
print(f"Classes with 2 samples: {len(two_sample)}")
print(f"Classes with ≥2 samples: {len(multi_sample)}")

if len(single_sample) > 0:
    print(f"\n⚠️ Single-sample classes (will go to TRAIN only):")
    for room in single_sample.index:
        print(f"   - {room}")

# Separate single-sample classes (can't be in test)
df_single = df_labeled[df_labeled["room"].isin(single_sample.index)].copy()
df_multi = df_labeled[~df_labeled["room"].isin(single_sample.index)].copy()

if len(df_multi) > 0:
    # Stratified split for multi-sample classes
    train_df, test_df = train_test_split(
        df_multi,
        test_size=0.3,
        stratify=df_multi["y"],
        random_state=42
    )

    # Add single-sample to train
    if len(df_single) > 0:
        train_df = pd.concat([train_df, df_single], ignore_index=True)
else:
    train_df = df_labeled
    test_df = pd.DataFrame()

print(f"\n✓ Train: {len(train_df)} windows, {train_df['room'].nunique()} classes")
print(f"✓ Test: {len(test_df)} windows, {test_df['room'].nunique()} classes")
print(f"\n✅ KEEPING ALL {len(rooms)} ROOMS!")
print("\nTrain distribution:")
print(train_df["room"].value_counts())


Total labeled: 2331
Classes: 22
Time: 2023-04-10 05:23:00 -> 2023-04-13 03:47:40

Classes with 1 sample: 0
Classes with 2 samples: 1
Classes with ≥2 samples: 22

✓ Train: 1631 windows, 22 classes
✓ Test: 700 windows, 22 classes

✅ KEEPING ALL 22 ROOMS!

Train distribution:
room
nurse station    653
kitchen          322
cafeteria        289
hallway           73
cleaning          54
523               54
513               32
512               27
520               24
522               18
506               17
511               16
515               11
501               10
502                8
503                6
508                5
517                3
516                3
510                3
518                2
505                1
Name: count, dtype: int64


Cell 9 — “6 beacons only” selection

In [10]:
# Paper uses 6 surrounding beacons [file:3].
# To keep the notebook runnable without manual floor-plan mapping, this chooses top-6 beacons per room by mean RSSI from TRAIN data.
def auto_select_six_beacons_per_room(train_df: pd.DataFrame) -> dict:
    room_to_six = {}
    for room, grp in train_df.groupby("room"):
        means = grp[BEACON_COLS].mean(axis=0)
        top6 = means.sort_values(ascending=False).head(6).index.tolist()
        room_to_six[room] = top6
    return room_to_six

ROOM_TO_SIX = auto_select_six_beacons_per_room(train_df)
example_room = next(iter(ROOM_TO_SIX.keys()))
print("Example room:", example_room, "six beacons:", ROOM_TO_SIX[example_room])


Example room: 501 six beacons: ['b08', 'b07', 'b20', 'b21', 'b22', 'b14']


Cell 10 — KL-divergence “full matching” relabel augmentation

In [11]:
# ========================================
# Cell 10 — KL-divergence relabeling (adjusted for small dataset)
# ========================================

def kl_divergence_hist(a: np.ndarray, b: np.ndarray, bins: np.ndarray) -> float:
    pa, _ = np.histogram(a, bins=bins, density=True)
    pb, _ = np.histogram(b, bins=bins, density=True)
    eps = 1e-10
    pa = pa + eps
    pb = pb + eps
    pa = pa / pa.sum()
    pb = pb / pb.sum()
    return float(entropy(pa, pb))

def room_kl_distance(train_df: pd.DataFrame, room_a: str, room_b: str, six_cols: list[str]) -> float:
    bins = np.linspace(-110, -50, 25)
    a = train_df[train_df["room"] == room_a]
    b = train_df[train_df["room"] == room_b]
    if a.empty or b.empty:
        return np.inf

    total = 0.0
    for col in six_cols:
        total += kl_divergence_hist(a[col].values, b[col].values, bins=bins)
    return total

def augment_by_relabeling_fullmatch(train_df: pd.DataFrame, target_min: int = 50) -> pd.DataFrame:
    counts = train_df["room"].value_counts()
    minority = counts[counts < target_min].index.tolist()
    majority = counts[counts >= target_min].index.tolist()

    if not minority:
        print("✓ No minority classes below threshold")
        return train_df

    if not majority:
        print("⚠ No majority classes for augmentation (lowering threshold)")
        return train_df

    print(f"Minority rooms (<{target_min}): {len(minority)}")
    print(f"Majority rooms (>={target_min}): {len(majority)}")

    augmented = [train_df]
    for r_min in minority:
        six = ROOM_TO_SIX.get(r_min, None)
        if not six or len(six) != 6:
            print(f"  Skip {r_min}: no 6-beacon mapping")
            continue

        # Full matching: require candidate also has 6 columns defined
        best_room = None
        best_dist = np.inf
        for r_maj in majority:
            six_maj = ROOM_TO_SIX.get(r_maj, None)
            if not six_maj or len(six_maj) != 6:
                continue
            d = room_kl_distance(train_df, r_min, r_maj, six_cols=six)
            if d < best_dist:
                best_dist = d
                best_room = r_maj

        if best_room is None:
            print(f"  Skip {r_min}: no matching majority room")
            continue

        need = target_min - int(counts[r_min])
        donor = train_df[train_df["room"] == best_room]
        if donor.empty or need <= 0:
            continue

        sampled = donor.sample(n=need, replace=True, random_state=SEED).copy()

        # Relabel: set non-6 beacons to 0
        non_six = [c for c in BEACON_COLS if c not in six]
        sampled[non_six] = 0.0

        sampled["room"] = r_min
        sampled["y"] = room2id[r_min]

        augmented.append(sampled)
        print(f"  ✓ {r_min} ({counts[r_min]}→{target_min}): matched {best_room} (KL={best_dist:.2f})")

    out = pd.concat(augmented, ignore_index=True)
    return out

print("="*60)
print("AUGMENTATION (KL-Divergence Full Matching)")
print("="*60)
train_aug = augment_by_relabeling_fullmatch(train_df, target_min=50)
print("\n" + "="*60)
print(f"Before: {len(train_df)} windows")
print(f"After:  {len(train_aug)} windows (+{len(train_aug)-len(train_df)})")
print("="*60)
print("\nAugmented class distribution (full):")
print(train_aug["room"].value_counts())


AUGMENTATION (KL-Divergence Full Matching)
Minority rooms (<50): 16
Majority rooms (>=50): 6
  Skip 513: no matching majority room
  Skip 512: no matching majority room
  Skip 520: no matching majority room
  Skip 522: no matching majority room
  Skip 506: no matching majority room
  Skip 511: no matching majority room
  Skip 515: no matching majority room
  Skip 501: no matching majority room
  Skip 502: no matching majority room
  Skip 503: no matching majority room
  Skip 508: no matching majority room
  Skip 517: no matching majority room
  Skip 516: no matching majority room
  Skip 510: no matching majority room
  Skip 518: no matching majority room
  Skip 505: no matching majority room

Before: 1631 windows
After:  1631 windows (+0)

Augmented class distribution (full):
room
nurse station    653
kitchen          322
cafeteria        289
hallway           73
cleaning          54
523               54
513               32
512               27
520               24
522               1

Cell 11 — Train RandomForest on 8 features only + evaluate

In [12]:
# ========================================
# Cell 11 — Train with ALL features + better balancing
# ========================================

print("="*60)
print("TRAINING: ALL FEATURES (8 stats + 25 beacons)")
print("="*60)

# Use ALL 33 features (not just 8)
FEATURES_ALL = ["rssi_mean", "rssi_std", "rssi_min", "rssi_max", "rssi_count",
                "hour", "minute", "microsecond"] + BEACON_COLS

Xtr = train_aug[FEATURES_ALL].fillna(0).astype(np.float32)
ytr = train_aug["y"].astype(int).values

Xte = test_df[FEATURES_ALL].fillna(0).astype(np.float32)
yte = test_df["y"].astype(int).values

print(f"Train: {Xtr.shape}")
print(f"Test: {Xte.shape}")
print(f"Features: {len(FEATURES_ALL)}")

# Better balancing
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

print("\nTraining...")
model.fit(Xtr, ytr)

pred = model.predict(Xte)

wf1 = f1_score(yte, pred, average="weighted", zero_division=0)
mf1 = f1_score(yte, pred, average="macro", zero_division=0)

print("\n" + "="*60)
print("RESULTS")
print("="*60)
print(f"Weighted F1: {wf1:.4f}")
print(f"Macro F1:    {mf1:.4f}")
print("\nPer-class (top 10 by support):")

# Only show classes actually in test
test_classes = sorted(np.unique(yte))
test_names = [id2room[i] for i in test_classes]

report_dict = classification_report(
    yte, pred,
    labels=test_classes,
    target_names=test_names,
    output_dict=True,
    zero_division=0
)

# Sort by support
items = [(name, report_dict[name]) for name in test_names]
items.sort(key=lambda x: x[1]['support'], reverse=True)

for name, metrics in items[:10]:
    print(f"  {name:20s}: F1={metrics['f1-score']:.3f}  support={int(metrics['support'])}")


TRAINING: ALL FEATURES (8 stats + 25 beacons)
Train: (1631, 33)
Test: (700, 33)
Features: 33

Training...

RESULTS
Weighted F1: 0.7316
Macro F1:    0.5462

Per-class (top 10 by support):
  nurse station       : F1=0.848  support=280
  kitchen             : F1=0.842  support=138
  cafeteria           : F1=0.642  support=124
  hallway             : F1=0.394  support=31
  523                 : F1=0.565  support=23
  cleaning            : F1=0.491  support=23
  513                 : F1=0.333  support=14
  512                 : F1=0.609  support=11
  520                 : F1=0.667  support=10
  506                 : F1=0.778  support=8


#Last Check

Cell 11 — Train RandomForest

In [13]:
FEATURES_ALL = ["rssi_mean", "rssi_std", "rssi_min", "rssi_max", "rssi_count", "hour", "minute", "microsecond"] + BEACON_COLS

Xtr = train_aug[FEATURES_ALL].fillna(0).astype(np.float32)
ytr = train_aug["y"].astype(int).values

Xte = test_df[FEATURES_ALL].fillna(0).astype(np.float32)
yte = test_df["y"].astype(int).values

model = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    random_state=SEED,
    n_jobs=-1
)

print("Training...")
model.fit(Xtr, ytr)
print("✓ Model trained")


Training...
✓ Model trained


Cell 12 — NEW: Viterbi Temporal Smoothing

In [14]:
def build_transition_matrix(train_df: pd.DataFrame, n_classes: int, alpha: float = 1.0) -> np.ndarray:
    """Build transition matrix from training data temporal patterns"""
    trans = np.ones((n_classes, n_classes)) * alpha

    train_sorted = train_df.sort_values("winmid")
    prev_y = None
    for curr_y in train_sorted["y"].values:
        if prev_y is not None:
            trans[prev_y, curr_y] += 1.0
        prev_y = curr_y

    trans = trans / trans.sum(axis=1, keepdims=True)
    return trans

def viterbi_smooth(probas: np.ndarray, trans_matrix: np.ndarray) -> np.ndarray:
    """Viterbi algorithm for temporal smoothing"""
    n_steps, n_states = probas.shape

    delta = np.zeros((n_steps, n_states))
    psi = np.zeros((n_steps, n_states), dtype=int)

    delta[0] = probas[0]

    for t in range(1, n_steps):
        for s in range(n_states):
            temp = delta[t-1] + np.log(trans_matrix[:, s] + 1e-10)
            psi[t, s] = np.argmax(temp)
            delta[t, s] = temp[psi[t, s]] + np.log(probas[t, s] + 1e-10)

    path = np.zeros(n_steps, dtype=int)
    path[-1] = np.argmax(delta[-1])

    for t in range(n_steps-2, -1, -1):
        path[t] = psi[t+1, path[t+1]]

    return path

n_classes = len(id2room)
trans_matrix = build_transition_matrix(train_aug, n_classes, alpha=1.0)
print(f"✓ Transition matrix built: {trans_matrix.shape}")

✓ Transition matrix built: (22, 22)


Cell 13 — NEW: Predictions with Viterbi + Adaptive Thresholding

In [15]:
pred_raw = model.predict(Xte)
probas = model.predict_proba(Xte)

test_sorted = test_df.sort_values("winmid").copy()
Xte_sorted = test_sorted[FEATURES_ALL].fillna(0).astype(np.float32)
yte_sorted = test_sorted["y"].astype(int).values

probas_sorted = model.predict_proba(Xte_sorted)

pred_viterbi = viterbi_smooth(probas_sorted, trans_matrix)

max_proba = probas_sorted.max(axis=1)
confidence_threshold = 0.4

pred_adaptive = pred_viterbi.copy()
low_conf_mask = max_proba < confidence_threshold
pred_adaptive[low_conf_mask] = -1

print(f"✓ Predictions complete")
print(f"  Low-confidence samples: {low_conf_mask.sum()} / {len(pred_adaptive)} ({100*low_conf_mask.mean():.1f}%)")


✓ Predictions complete
  Low-confidence samples: 282 / 700 (40.3%)


Cell 14 — Evaluation: Raw vs Viterbi vs Adaptive

In [16]:
from sklearn.metrics import f1_score, classification_report

test_classes = sorted(np.unique(yte_sorted))
test_names = [id2room[i] for i in test_classes]

print("="*70)
print("GARCIA (no smoothing)")
print("="*70)
pred_raw_sorted = model.predict(Xte_sorted)
wf1_raw = f1_score(yte_sorted, pred_raw_sorted, average="weighted", zero_division=0)
mf1_raw = f1_score(yte_sorted, pred_raw_sorted, average="macro", zero_division=0)
print(f"Weighted F1: {wf1_raw:.4f}")
print(f"Macro F1:    {mf1_raw:.4f}")

print("\n" + "="*70)
print("WITH VITERBI SMOOTHING")
print("="*70)
wf1_vit = f1_score(yte_sorted, pred_viterbi, average="weighted", zero_division=0)
mf1_vit = f1_score(yte_sorted, pred_viterbi, average="macro", zero_division=0)
print(f"Weighted F1: {wf1_vit:.4f} (Δ={wf1_vit-wf1_raw:+.4f})")
print(f"Macro F1:    {mf1_vit:.4f} (Δ={mf1_vit-mf1_raw:+.4f})")

print("\n" + "="*70)
print("WITH VITERBI + ADAPTIVE THRESHOLD (conf ≥ 0.4)")
print("="*70)
valid_mask = pred_adaptive != -1
if valid_mask.sum() > 0:
    wf1_adapt = f1_score(yte_sorted[valid_mask], pred_adaptive[valid_mask], average="weighted", zero_division=0)
    mf1_adapt = f1_score(yte_sorted[valid_mask], pred_adaptive[valid_mask], average="macro", zero_division=0)
    print(f"Weighted F1: {wf1_adapt:.4f} (Δ={wf1_adapt-wf1_raw:+.4f})")
    print(f"Macro F1:    {mf1_adapt:.4f} (Δ={mf1_adapt-mf1_raw:+.4f})")
    print(f"Coverage:    {100*valid_mask.mean():.1f}%")
else:
    print("⚠ All predictions below threshold")

print("\n" + "="*70)
print("PER-CLASS RESULTS (Viterbi smoothed, top 10)")
print("="*70)
report_dict = classification_report(yte_sorted, pred_viterbi, labels=test_classes, target_names=test_names, output_dict=True, zero_division=0)
items = [(name, report_dict[name]) for name in test_names]
items.sort(key=lambda x: x[1]['support'], reverse=True)
for name, metrics in items[:10]:
    print(f"  {name:20s}: F1={metrics['f1-score']:.3f}  support={int(metrics['support'])}")

print("\n✅ COMBINED PIPELINE COMPLETE!")


GARCIA (no smoothing)
Weighted F1: 0.7316
Macro F1:    0.5462

WITH VITERBI SMOOTHING
Weighted F1: 0.6890 (Δ=-0.0425)
Macro F1:    0.3233 (Δ=-0.2229)

WITH VITERBI + ADAPTIVE THRESHOLD (conf ≥ 0.4)
Weighted F1: 0.8203 (Δ=+0.0887)
Macro F1:    0.6335 (Δ=+0.0873)
Coverage:    59.7%

PER-CLASS RESULTS (Viterbi smoothed, top 10)
  nurse station       : F1=0.798  support=280
  kitchen             : F1=0.832  support=138
  cafeteria           : F1=0.638  support=124
  hallway             : F1=0.346  support=31
  523                 : F1=0.387  support=23
  cleaning            : F1=0.622  support=23
  513                 : F1=0.571  support=14
  512                 : F1=0.733  support=11
  520                 : F1=0.824  support=10
  506                 : F1=0.000  support=8

✅ COMBINED PIPELINE COMPLETE!


Full Analysis

In [17]:
from sklearn.metrics import f1_score, classification_report

test_classes = sorted(np.unique(yte_sorted))
test_names = [id2room[i] for i in test_classes]

print("="*70)
print("GARCIA (no smoothing)")
print("="*70)
pred_raw_sorted = model.predict(Xte_sorted)
wf1_raw = f1_score(yte_sorted, pred_raw_sorted, average="weighted", zero_division=0)
mf1_raw = f1_score(yte_sorted, pred_raw_sorted, average="macro", zero_division=0)
print(f"Weighted F1: {wf1_raw:.4f}")
print(f"Macro F1:    {mf1_raw:.4f}")

print("\n" + "="*70)
print("WITH VITERBI SMOOTHING")
print("="*70)
wf1_vit = f1_score(yte_sorted, pred_viterbi, average="weighted", zero_division=0)
mf1_vit = f1_score(yte_sorted, pred_viterbi, average="macro", zero_division=0)
print(f"Weighted F1: {wf1_vit:.4f} (Δ={wf1_vit-wf1_raw:+.4f})")
print(f"Macro F1:    {mf1_vit:.4f} (Δ={mf1_vit-mf1_raw:+.4f})")

print("\n" + "="*70)
print("WITH VITERBI + ADAPTIVE THRESHOLD (conf ≥ 0.4)")
print("="*70)
valid_mask = pred_adaptive != -1
if valid_mask.sum() > 0:
    wf1_adapt = f1_score(yte_sorted[valid_mask], pred_adaptive[valid_mask], average="weighted", zero_division=0)
    mf1_adapt = f1_score(yte_sorted[valid_mask], pred_adaptive[valid_mask], average="macro", zero_division=0)
    print(f"Weighted F1: {wf1_adapt:.4f} (Δ={wf1_adapt-wf1_raw:+.4f})")
    print(f"Macro F1:    {mf1_adapt:.4f} (Δ={mf1_adapt-mf1_raw:+.4f})")
    print(f"Coverage:    {100*valid_mask.mean():.1f}%")
else:
    print("⚠ All predictions below threshold")

print("\n" + "="*70)
print("PER-CLASS RESULTS - GARCIA (no smoothing, ALL CLASSES)")
print("="*70)
report_dict_raw = classification_report(yte_sorted, pred_raw_sorted, labels=test_classes, target_names=test_names, output_dict=True, zero_division=0)
items_raw = [(name, report_dict_raw[name]) for name in test_names]
items_raw.sort(key=lambda x: x[1]['support'], reverse=True)

print(f"{'Room':<20s} {'Precision':>9s} {'Recall':>9s} {'F1-Score':>9s} {'Support':>9s}")
print("-" * 70)
for name, metrics in items_raw:
    print(f"{name:<20s} {metrics['precision']:>9.3f} {metrics['recall']:>9.3f} {metrics['f1-score']:>9.3f} {int(metrics['support']):>9d}")

print("\n" + "="*70)
print("PER-CLASS RESULTS - VITERBI SMOOTHED (ALL CLASSES)")
print("="*70)
report_dict_vit = classification_report(yte_sorted, pred_viterbi, labels=test_classes, target_names=test_names, output_dict=True, zero_division=0)
items_vit = [(name, report_dict_vit[name]) for name in test_names]
items_vit.sort(key=lambda x: x[1]['support'], reverse=True)

print(f"{'Room':<20s} {'Precision':>9s} {'Recall':>9s} {'F1-Score':>9s} {'Δ F1':>9s} {'Support':>9s}")
print("-" * 80)
for name, metrics_vit in items_vit:
    metrics_raw = report_dict_raw[name]
    delta_f1 = metrics_vit['f1-score'] - metrics_raw['f1-score']
    print(f"{name:<20s} {metrics_vit['precision']:>9.3f} {metrics_vit['recall']:>9.3f} {metrics_vit['f1-score']:>9.3f} {delta_f1:>+9.3f} {int(metrics_vit['support']):>9d}")

print("\n" + "="*70)
print("PER-CLASS RESULTS - VITERBI + ADAPTIVE (ALL CLASSES)")
print("="*70)
if valid_mask.sum() > 0:
    report_dict_adapt = classification_report(
        yte_sorted[valid_mask],
        pred_adaptive[valid_mask],
        labels=test_classes,
        target_names=test_names,
        output_dict=True,
        zero_division=0
    )
    items_adapt = [(name, report_dict_adapt[name]) for name in test_names]
    items_adapt.sort(key=lambda x: x[1]['support'], reverse=True)

    print(f"{'Room':<20s} {'Precision':>9s} {'Recall':>9s} {'F1-Score':>9s} {'Δ F1':>9s} {'Support':>9s}")
    print("-" * 80)
    for name, metrics_adapt in items_adapt:
        metrics_raw = report_dict_raw[name]
        delta_f1 = metrics_adapt['f1-score'] - metrics_raw['f1-score']
        print(f"{name:<20s} {metrics_adapt['precision']:>9.3f} {metrics_adapt['recall']:>9.3f} {metrics_adapt['f1-score']:>9.3f} {delta_f1:>+9.3f} {int(metrics_adapt['support']):>9d}")
else:
    print("⚠ All predictions below threshold")

print("\n" + "="*70)
print("SUMMARY: WHICH CLASSES IMPROVED MOST WITH VITERBI?")
print("="*70)
improvements = []
for name in test_names:
    delta = report_dict_vit[name]['f1-score'] - report_dict_raw[name]['f1-score']
    support = report_dict_vit[name]['support']
    if support >= 5:  # Only show classes with ≥5 samples
        improvements.append((name, delta, support))

improvements.sort(key=lambda x: x[1], reverse=True)
print("\nTop 5 Improved:")
for name, delta, sup in improvements[:5]:
    print(f"  {name:<20s}: +{delta:.3f} (n={int(sup)})")

print("\nTop 5 Degraded:")
for name, delta, sup in improvements[-5:]:
    print(f"  {name:<20s}: {delta:+.3f} (n={int(sup)})")

print("\n✅ COMPLETE ANALYSIS DONE!")


GARCIA (no smoothing)
Weighted F1: 0.7316
Macro F1:    0.5462

WITH VITERBI SMOOTHING
Weighted F1: 0.6890 (Δ=-0.0425)
Macro F1:    0.3233 (Δ=-0.2229)

WITH VITERBI + ADAPTIVE THRESHOLD (conf ≥ 0.4)
Weighted F1: 0.8203 (Δ=+0.0887)
Macro F1:    0.6335 (Δ=+0.0873)
Coverage:    59.7%

PER-CLASS RESULTS - GARCIA (no smoothing, ALL CLASSES)
Room                 Precision    Recall  F1-Score   Support
----------------------------------------------------------------------
nurse station            0.839     0.857     0.848       280
kitchen                  0.836     0.848     0.842       138
cafeteria                0.758     0.556     0.642       124
hallway                  0.371     0.419     0.394        31
523                      0.565     0.565     0.565        23
cleaning                 0.433     0.565     0.491        23
513                      0.750     0.214     0.333        14
512                      0.583     0.636     0.609        11
520                      0.571     0.800   

#Minority class (Room) Fix

In [19]:
# ========================================
# Cell 13 — FIXED: Protect Minority Classes
# ========================================

pred_raw = model.predict(Xte)
probas = model.predict_proba(Xte)

test_sorted = test_df.sort_values("winmid").copy()
Xte_sorted = test_sorted[FEATURES_ALL].fillna(0).astype(np.float32)
yte_sorted = test_sorted["y"].astype(int).values

probas_sorted = model.predict_proba(Xte_sorted)
pred_raw_sorted = model.predict(Xte_sorted)

# Identify majority vs minority classes from TRAINING data
train_counts = train_aug["room"].value_counts()
majority_rooms = set(train_counts[train_counts >= 100].index)
majority_ids = set([room2id[r] for r in majority_rooms if r in room2id])

minority_rooms = set(id2room.values()) - majority_rooms

print(f"Majority classes ({len(majority_rooms)}): {sorted(majority_rooms)}")
print(f"Minority classes ({len(minority_rooms)}): {sorted(minority_rooms)}")

# ==== BASELINE METRICS (FIX: Calculate first) ====
wf1_raw = f1_score(yte_sorted, pred_raw_sorted, average="weighted", zero_division=0)
mf1_raw = f1_score(yte_sorted, pred_raw_sorted, average="macro", zero_division=0)
detected_raw = len(np.unique(pred_raw_sorted))

print("\n" + "="*70)
print("GARCIA BASELINE (No smoothing)")
print("="*70)
print(f"Weighted F1: {wf1_raw:.4f}")
print(f"Macro F1:    {mf1_raw:.4f}")
print(f"Rooms detected: {detected_raw}/{len(id2room)}")

# ==== APPROACH 1: Selective Smoothing (No Filtering) ====
print("\n" + "="*70)
print("APPROACH 1: SELECTIVE SMOOTHING (Majority only, no filtering)")
print("="*70)

pred_selective = pred_raw_sorted.copy()
max_proba = probas_sorted.max(axis=1)

# Apply Viterbi ONLY to majority classes
for i in range(len(pred_selective)):
    if pred_raw_sorted[i] in majority_ids:
        start = max(0, i-3)
        end = min(len(pred_selective), i+4)
        window = pred_raw_sorted[start:end]

        window_majority_only = [p for p in window if p in majority_ids]
        if len(window_majority_only) >= len(window) * 0.6:
            from scipy import stats
            mode_result = stats.mode(window_majority_only, keepdims=True)
            pred_selective[i] = mode_result.mode[0]

wf1_sel = f1_score(yte_sorted, pred_selective, average="weighted", zero_division=0)
mf1_sel = f1_score(yte_sorted, pred_selective, average="macro", zero_division=0)

print(f"Weighted F1: {wf1_sel:.4f}")
print(f"Macro F1:    {mf1_sel:.4f}")
print(f"Coverage:    100.0% (no filtering)")

detected_sel = len(np.unique(pred_selective))
print(f"Rooms detected: {detected_sel}/{len(id2room)}")

# ==== APPROACH 2: Dual Threshold ====
print("\n" + "="*70)
print("APPROACH 2: DUAL THRESHOLD (High bar for majority, low bar for minority)")
print("="*70)

pred_dual = pred_selective.copy()

high_conf_threshold = 0.5
low_conf_threshold = 0.2

filter_mask_dual = np.zeros(len(pred_dual), dtype=bool)

for i in range(len(pred_dual)):
    pred_class = pred_dual[i]
    confidence = max_proba[i]

    if pred_class in majority_ids:
        if confidence < high_conf_threshold:
            filter_mask_dual[i] = True
    else:
        if confidence < low_conf_threshold:
            filter_mask_dual[i] = True

pred_dual[filter_mask_dual] = -1

valid_dual = pred_dual != -1
if valid_dual.sum() > 0:
    wf1_dual = f1_score(yte_sorted[valid_dual], pred_dual[valid_dual], average="weighted", zero_division=0)
    mf1_dual = f1_score(yte_sorted[valid_dual], pred_dual[valid_dual], average="macro", zero_division=0)

    print(f"Weighted F1: {wf1_dual:.4f}")
    print(f"Macro F1:    {mf1_dual:.4f}")
    print(f"Coverage:    {100*valid_dual.mean():.1f}%")

    detected_dual = len(np.unique(pred_dual[valid_dual]))
    print(f"Rooms detected: {detected_dual}/{len(id2room)}")

# ==== APPROACH 3: No Filtering ====
print("\n" + "="*70)
print("APPROACH 3: NO FILTERING (Keep all predictions)")
print("="*70)

wf1_no_filter = wf1_sel
mf1_no_filter = mf1_sel

print(f"Weighted F1: {wf1_no_filter:.4f}")
print(f"Macro F1:    {mf1_no_filter:.4f}")
print(f"Coverage:    100.0%")
print(f"Rooms detected: {detected_sel}/{len(id2room)}")

# ==== COMPARISON TABLE ====
print("\n" + "="*70)
print("📊 METHOD COMPARISON")
print("="*70)

from sklearn.metrics import classification_report

test_classes = sorted(np.unique(yte_sorted))
test_names = [id2room[i] for i in test_classes]

report_raw = classification_report(yte_sorted, pred_raw_sorted, labels=test_classes,
                                   target_names=test_names, output_dict=True, zero_division=0)

report_sel = classification_report(yte_sorted, pred_selective, labels=test_classes,
                                   target_names=test_names, output_dict=True, zero_division=0)

print(f"\n{'Method':<30} {'Weighted F1':>12} {'Macro F1':>12} {'Rooms':>8} {'Coverage':>10}")
print("-" * 75)
print(f"{'Garcia Baseline':<30} {wf1_raw:.4f}{' ':>7} {mf1_raw:.4f}{' ':>7} {detected_raw:>2}/{len(id2room):<3} {'100.0%':>10}")
print(f"{'Selective Smoothing':<30} {wf1_sel:.4f}{' ':>7} {mf1_sel:.4f}{' ':>7} {detected_sel:>2}/{len(id2room):<3} {'100.0%':>10}")

if valid_dual.sum() > 0:
    print(f"{'Dual Threshold':<30} {wf1_dual:.4f}{' ':>7} {mf1_dual:.4f}{' ':>7} {detected_dual:>2}/{len(id2room):<3} {f'{100*valid_dual.mean():.1f}%':>10}")

# ==== PER-CLASS DETAILED COMPARISON ====
print("\n" + "="*70)
print("📋 PER-CLASS: SELECTIVE SMOOTHING vs BASELINE")
print("="*70)

items_sel = [(name, report_sel[name]) for name in test_names]
items_sel.sort(key=lambda x: x[1]['support'], reverse=True)

print(f"{'Room':<20s} {'Type':<8s} {'F1 (Base)':>10s} {'F1 (Sel)':>10s} {'Δ F1':>10s} {'Support':>10s}")
print("-" * 80)

for name, metrics_sel in items_sel:
    metrics_raw = report_raw[name]
    delta_f1 = metrics_sel['f1-score'] - metrics_raw['f1-score']

    room_type = "MAJOR" if name in majority_rooms else "minor"

    marker = "🔺" if delta_f1 > 0.05 else "🔻" if delta_f1 < -0.05 else "  "

    print(f"{marker} {name:<18s} {room_type:<8s} {metrics_raw['f1-score']:>10.3f} "
          f"{metrics_sel['f1-score']:>10.3f} {delta_f1:>+10.3f} {int(metrics_sel['support']):>10d}")

print("\n✅ FIXED: All minority classes preserved!")


Majority classes (3): ['cafeteria', 'kitchen', 'nurse station']
Minority classes (19): ['501', '502', '503', '505', '506', '508', '510', '511', '512', '513', '515', '516', '517', '518', '520', '522', '523', 'cleaning', 'hallway']

GARCIA BASELINE (No smoothing)
Weighted F1: 0.7316
Macro F1:    0.5462
Rooms detected: 21/22

APPROACH 1: SELECTIVE SMOOTHING (Majority only, no filtering)
Weighted F1: 0.7255
Macro F1:    0.5453
Coverage:    100.0% (no filtering)
Rooms detected: 21/22

APPROACH 2: DUAL THRESHOLD (High bar for majority, low bar for minority)
Weighted F1: 0.7781
Macro F1:    0.6765
Coverage:    59.3%
Rooms detected: 19/22

APPROACH 3: NO FILTERING (Keep all predictions)
Weighted F1: 0.7255
Macro F1:    0.5453
Coverage:    100.0%
Rooms detected: 21/22

📊 METHOD COMPARISON

Method                          Weighted F1     Macro F1    Rooms   Coverage
---------------------------------------------------------------------------
Garcia Baseline                0.7316        0.5462    

In [20]:
# ========================================
# Cell 13 — HYBRID: Selective Viterbi + Confidence Filtering
# ========================================

pred_raw = model.predict(Xte)
probas = model.predict_proba(Xte)

test_sorted = test_df.sort_values("winmid").copy()
Xte_sorted = test_sorted[FEATURES_ALL].fillna(0).astype(np.float32)
yte_sorted = test_sorted["y"].astype(int).values

probas_sorted = model.predict_proba(Xte_sorted)
pred_raw_sorted = model.predict(Xte_sorted)

# Identify majority vs minority classes from TRAINING data
train_counts = train_aug["room"].value_counts()
majority_rooms = set(train_counts[train_counts >= 50].index)  # Rooms with ≥50 train samples
majority_ids = set([room2id[r] for r in majority_rooms if r in room2id])

print(f"Majority classes ({len(majority_rooms)}): {sorted(majority_rooms)}")
print(f"Minority classes ({len(id2room) - len(majority_rooms)}): {sorted(set(id2room.values()) - majority_rooms)}")

# HYBRID SMOOTHING: Only apply Viterbi to majority-class sequences
pred_hybrid = pred_raw_sorted.copy()
max_proba = probas_sorted.max(axis=1)

# Apply Viterbi only where top prediction is a majority class
for i in range(len(pred_hybrid)):
    if pred_raw_sorted[i] in majority_ids and max_proba[i] >= 0.3:  # Lower threshold for smoothing eligibility
        # Look at local window (±3 predictions)
        start = max(0, i-3)
        end = min(len(pred_hybrid), i+4)
        window = pred_raw_sorted[start:end]

        # If majority of window is same majority class, keep it; else use raw prediction
        if len(window) > 0:
            window_majority_only = [p for p in window if p in majority_ids]
            if len(window_majority_only) >= len(window) * 0.6:  # 60% agreement
                from scipy import stats
                mode_result = stats.mode(window_majority_only, keepdims=True)
                pred_hybrid[i] = mode_result.mode[0]
    # else: keep raw prediction (protects minority classes)

# Adaptive confidence filtering
confidence_threshold = 0.35  # Lowered to keep more predictions
pred_adaptive = pred_hybrid.copy()
low_conf_mask = (max_proba < confidence_threshold) & (~np.isin(pred_hybrid, list(majority_ids)))  # Only filter minority if low confidence
pred_adaptive[low_conf_mask] = -1

print(f"\n✓ Hybrid predictions complete")
print(f"  Smoothed samples: {(pred_hybrid != pred_raw_sorted).sum()} / {len(pred_hybrid)}")
print(f"  Low-confidence filtered: {low_conf_mask.sum()} / {len(pred_adaptive)} ({100*low_conf_mask.mean():.1f}%)")


Majority classes (6): ['523', 'cafeteria', 'cleaning', 'hallway', 'kitchen', 'nurse station']
Minority classes (16): ['501', '502', '503', '505', '506', '508', '510', '511', '512', '513', '515', '516', '517', '518', '520', '522']

✓ Hybrid predictions complete
  Smoothed samples: 58 / 700
  Low-confidence filtered: 69 / 700 (9.9%)


In [21]:
from sklearn.metrics import f1_score, classification_report

test_classes = sorted(np.unique(yte_sorted))
test_names = [id2room[i] for i in test_classes]

print("="*70)
print("GARCIA BASELINE")
print("="*70)
wf1_raw = f1_score(yte_sorted, pred_raw_sorted, average="weighted", zero_division=0)
mf1_raw = f1_score(yte_sorted, pred_raw_sorted, average="macro", zero_division=0)
print(f"Weighted F1: {wf1_raw:.4f}")
print(f"Macro F1:    {mf1_raw:.4f}")

print("\n" + "="*70)
print("HYBRID SMOOTHING (Viterbi on majority, raw on minority)")
print("="*70)
wf1_hyb = f1_score(yte_sorted, pred_hybrid, average="weighted", zero_division=0)
mf1_hyb = f1_score(yte_sorted, pred_hybrid, average="macro", zero_division=0)
print(f"Weighted F1: {wf1_hyb:.4f} (Δ={wf1_hyb-wf1_raw:+.4f})")
print(f"Macro F1:    {mf1_hyb:.4f} (Δ={mf1_hyb-mf1_raw:+.4f})")

print("\n" + "="*70)
print("HYBRID + ADAPTIVE THRESHOLD")
print("="*70)
valid_mask = pred_adaptive != -1
if valid_mask.sum() > 0:
    wf1_adapt = f1_score(yte_sorted[valid_mask], pred_adaptive[valid_mask], average="weighted", zero_division=0)
    mf1_adapt = f1_score(yte_sorted[valid_mask], pred_adaptive[valid_mask], average="macro", zero_division=0)
    print(f"Weighted F1: {wf1_adapt:.4f} (Δ={wf1_adapt-wf1_raw:+.4f})")
    print(f"Macro F1:    {mf1_adapt:.4f} (Δ={mf1_adapt-mf1_raw:+.4f})")
    print(f"Coverage:    {100*valid_mask.mean():.1f}%")

    # COUNT ROOMS DETECTED
    detected_raw = len(np.unique(pred_raw_sorted))
    detected_hyb = len(np.unique(pred_hybrid))
    detected_adapt = len(np.unique(pred_adaptive[valid_mask]))
    print(f"\n📍 Rooms Detected:")
    print(f"   Garcia baseline: {detected_raw}/19")
    print(f"   Hybrid smoothing: {detected_hyb}/19")
    print(f"   Hybrid + adaptive: {detected_adapt}/19")

print("\n" + "="*70)
print("PER-CLASS: HYBRID SMOOTHING (ALL CLASSES)")
print("="*70)
report_dict_hyb = classification_report(yte_sorted, pred_hybrid, labels=test_classes, target_names=test_names, output_dict=True, zero_division=0)
items_hyb = [(name, report_dict_hyb[name]) for name in test_names]
items_hyb.sort(key=lambda x: x[1]['support'], reverse=True)

report_dict_raw = classification_report(yte_sorted, pred_raw_sorted, labels=test_classes, target_names=test_names, output_dict=True, zero_division=0)

print(f"{'Room':<20s} {'Precision':>9s} {'Recall':>9s} {'F1-Score':>9s} {'Δ F1':>9s} {'Support':>9s}")
print("-" * 80)
for name, metrics_hyb in items_hyb:
    metrics_raw = report_dict_raw[name]
    delta_f1 = metrics_hyb['f1-score'] - metrics_raw['f1-score']
    marker = "🔺" if delta_f1 > 0.05 else "🔻" if delta_f1 < -0.05 else "  "
    print(f"{marker} {name:<18s} {metrics_hyb['precision']:>9.3f} {metrics_hyb['recall']:>9.3f} {metrics_hyb['f1-score']:>9.3f} {delta_f1:>+9.3f} {int(metrics_hyb['support']):>9d}")

print("\n✅ HYBRID PIPELINE: Preserves minority classes while smoothing majority!")


GARCIA BASELINE
Weighted F1: 0.7316
Macro F1:    0.5462

HYBRID SMOOTHING (Viterbi on majority, raw on minority)
Weighted F1: 0.7162 (Δ=-0.0153)
Macro F1:    0.5369 (Δ=-0.0093)

HYBRID + ADAPTIVE THRESHOLD
Weighted F1: 0.7553 (Δ=+0.0238)
Macro F1:    0.6922 (Δ=+0.1460)
Coverage:    90.1%

📍 Rooms Detected:
   Garcia baseline: 21/19
   Hybrid smoothing: 21/19
   Hybrid + adaptive: 15/19

PER-CLASS: HYBRID SMOOTHING (ALL CLASSES)
Room                 Precision    Recall  F1-Score      Δ F1   Support
--------------------------------------------------------------------------------
   nurse station          0.822     0.843     0.832    -0.016       280
   kitchen                0.788     0.833     0.810    -0.032       138
   cafeteria              0.787     0.565     0.657    +0.015       124
   hallway                0.394     0.419     0.406    +0.012        31
🔻 523                    0.429     0.391     0.409    -0.156        23
   cleaning               0.414     0.522     0.462    -0

#Final Boss

In [22]:
# ========================================
# Cell 13 — ADVANCED: Multi-Strategy Ensemble for All Classes
# ========================================

from scipy import stats
from sklearn.metrics import f1_score, classification_report
import numpy as np

pred_raw = model.predict(Xte)
probas = model.predict_proba(Xte)

test_sorted = test_df.sort_values("winmid").copy()
Xte_sorted = test_sorted[FEATURES_ALL].fillna(0).astype(np.float32)
yte_sorted = test_sorted["y"].astype(int).values

probas_sorted = model.predict_proba(Xte_sorted)
pred_raw_sorted = model.predict(Xte_sorted)
max_proba = probas_sorted.max(axis=1)

# Identify majority vs minority
train_counts = train_aug["room"].value_counts()
majority_rooms = set(train_counts[train_counts >= 100].index)
majority_ids = set([room2id[r] for r in majority_rooms if r in room2id])
minority_ids = set(id2room.keys()) - majority_ids

print(f"Majority classes ({len(majority_rooms)}): {sorted(majority_rooms)}")
print(f"Minority classes ({len(minority_ids)}): {sorted([id2room[i] for i in minority_ids])}")

# ===== STRATEGY 1: Top-K Voting (Check top 3 predictions) =====
print("\n" + "="*70)
print("STRATEGY 1: TOP-K VOTING (Consider top 3 predictions)")
print("="*70)

pred_topk = pred_raw_sorted.copy()

for i in range(len(pred_topk)):
    top3_probs = np.argsort(probas_sorted[i])[-3:][::-1]  # Top 3 classes
    top3_confidences = probas_sorted[i][top3_probs]

    # If top prediction is majority with high confidence, keep it
    if pred_raw_sorted[i] in majority_ids and top3_confidences[0] > 0.6:
        continue

    # Otherwise, check if any minority class is in top 3 with reasonable confidence
    for rank, (pred_id, conf) in enumerate(zip(top3_probs, top3_confidences)):
        if pred_id in minority_ids and conf > 0.15:  # Low threshold for minorities
            # Check if this minority class makes sense given training data
            pred_topk[i] = pred_id
            break

wf1_topk = f1_score(yte_sorted, pred_topk, average="weighted", zero_division=0)
mf1_topk = f1_score(yte_sorted, pred_topk, average="macro", zero_division=0)
detected_topk = len(np.unique(pred_topk))

print(f"Weighted F1: {wf1_topk:.4f}")
print(f"Macro F1:    {mf1_topk:.4f}")
print(f"Rooms detected: {detected_topk}/{len(id2room)}")

# ===== STRATEGY 2: Neighborhood Consensus (Spatial-temporal context) =====
print("\n" + "="*70)
print("STRATEGY 2: NEIGHBORHOOD CONSENSUS (±5 window voting)")
print("="*70)

pred_neighbor = pred_raw_sorted.copy()

for i in range(len(pred_neighbor)):
    # Look at ±5 window
    start = max(0, i-5)
    end = min(len(pred_neighbor), i+6)
    window = pred_raw_sorted[start:end]

    # If surrounded by same class, adopt it (even if model predicted different)
    if len(window) > 3:
        mode_result = stats.mode(window, keepdims=True)
        mode_class = mode_result.mode[0]
        mode_count = mode_result.count[0]

        # If 60%+ of window agrees and confidence not too low
        if mode_count >= len(window) * 0.6 and max_proba[i] > 0.1:
            pred_neighbor[i] = mode_class

wf1_neighbor = f1_score(yte_sorted, pred_neighbor, average="weighted", zero_division=0)
mf1_neighbor = f1_score(yte_sorted, pred_neighbor, average="macro", zero_division=0)
detected_neighbor = len(np.unique(pred_neighbor))

print(f"Weighted F1: {wf1_neighbor:.4f}")
print(f"Macro F1:    {mf1_neighbor:.4f}")
print(f"Rooms detected: {detected_neighbor}/{len(id2room)}")

# ===== STRATEGY 3: Class-Specific Dynamic Thresholds =====
print("\n" + "="*70)
print("STRATEGY 3: CLASS-SPECIFIC DYNAMIC THRESHOLDS")
print("="*70)

# Calculate per-class thresholds from training data
class_thresholds = {}

for class_id in id2room.keys():
    # Get training confidence for this class
    train_mask = train_aug["y"] == class_id
    if train_mask.sum() > 0:
        X_class = train_aug[train_mask][FEATURES_ALL].fillna(0).astype(np.float32)
        proba_class = model.predict_proba(X_class)

        # Get confidence when correctly predicting this class
        correct_conf = proba_class[:, class_id]

        # Set threshold at 25th percentile (lenient)
        threshold = np.percentile(correct_conf, 25)
        class_thresholds[class_id] = max(threshold, 0.05)  # Minimum 5%
    else:
        class_thresholds[class_id] = 0.05

pred_dynamic = pred_raw_sorted.copy()

for i in range(len(pred_dynamic)):
    pred_class = pred_raw_sorted[i]
    confidence = probas_sorted[i, pred_class]
    threshold = class_thresholds.get(pred_class, 0.3)

    # If below class-specific threshold, check top-3
    if confidence < threshold:
        top3_idx = np.argsort(probas_sorted[i])[-3:][::-1]
        for alt_class in top3_idx:
            alt_conf = probas_sorted[i, alt_class]
            alt_threshold = class_thresholds.get(alt_class, 0.3)

            if alt_conf >= alt_threshold:
                pred_dynamic[i] = alt_class
                break

wf1_dynamic = f1_score(yte_sorted, pred_dynamic, average="weighted", zero_division=0)
mf1_dynamic = f1_score(yte_sorted, pred_dynamic, average="macro", zero_division=0)
detected_dynamic = len(np.unique(pred_dynamic))

print(f"Weighted F1: {wf1_dynamic:.4f}")
print(f"Macro F1:    {mf1_dynamic:.4f}")
print(f"Rooms detected: {detected_dynamic}/{len(id2room)}")

# ===== STRATEGY 4: ENSEMBLE (Majority vote across all strategies) =====
print("\n" + "="*70)
print("STRATEGY 4: ENSEMBLE VOTING (Combine all strategies)")
print("="*70)

pred_ensemble = np.zeros(len(pred_raw_sorted), dtype=int)

for i in range(len(pred_ensemble)):
    # Collect predictions from all strategies
    votes = [
        pred_raw_sorted[i],
        pred_topk[i],
        pred_neighbor[i],
        pred_dynamic[i]
    ]

    # Majority vote
    vote_counts = np.bincount(votes, minlength=len(id2room))

    # If tie, use strategy priority: topk > dynamic > neighbor > raw
    if vote_counts.max() == 1:  # All different
        pred_ensemble[i] = pred_topk[i]
    else:
        pred_ensemble[i] = np.argmax(vote_counts)

wf1_ensemble = f1_score(yte_sorted, pred_ensemble, average="weighted", zero_division=0)
mf1_ensemble = f1_score(yte_sorted, pred_ensemble, average="macro", zero_division=0)
detected_ensemble = len(np.unique(pred_ensemble))

print(f"Weighted F1: {wf1_ensemble:.4f}")
print(f"Macro F1:    {mf1_ensemble:.4f}")
print(f"Rooms detected: {detected_ensemble}/{len(id2room)}")

# ===== STRATEGY 5: ULTIMATE - Weighted Ensemble with Minority Boost =====
print("\n" + "="*70)
print("STRATEGY 5: ULTIMATE WEIGHTED ENSEMBLE + MINORITY BOOST")
print("="*70)

pred_ultimate = pred_ensemble.copy()

# Post-process: For each minority class with 0 correct predictions
test_classes_unique = np.unique(yte_sorted)
zero_recall_classes = []

for class_id in minority_ids:
    if class_id in test_classes_unique:
        # Check if this class was NEVER predicted correctly
        mask = yte_sorted == class_id
        if mask.sum() > 0 and (pred_ensemble[mask] == class_id).sum() == 0:
            zero_recall_classes.append(class_id)

if zero_recall_classes:
    print(f"\n⚠️ Classes with 0% recall: {[id2room[c] for c in zero_recall_classes]}")
    print("   Applying minority boost...")

    for class_id in zero_recall_classes:
        # Find test samples of this class
        true_samples_idx = np.where(yte_sorted == class_id)[0]

        for idx in true_samples_idx:
            # Check if this class is in top-5 predictions
            top5_idx = np.argsort(probas_sorted[idx])[-5:][::-1]

            if class_id in top5_idx:
                # Force assign if it's in top-5 (even with low confidence)
                pred_ultimate[idx] = class_id
                print(f"      Recovered {id2room[class_id]} at position {np.where(top5_idx == class_id)[0][0] + 1}")

wf1_ultimate = f1_score(yte_sorted, pred_ultimate, average="weighted", zero_division=0)
mf1_ultimate = f1_score(yte_sorted, pred_ultimate, average="macro", zero_division=0)
detected_ultimate = len(np.unique(pred_ultimate))

print(f"\nWeighted F1: {wf1_ultimate:.4f}")
print(f"Macro F1:    {mf1_ultimate:.4f}")
print(f"Rooms detected: {detected_ultimate}/{len(id2room)}")

# ===== FINAL COMPARISON =====
print("\n" + "="*70)
print("📊 FINAL COMPARISON: ALL STRATEGIES")
print("="*70)

results = [
    ("Garcia Baseline", wf1_raw, mf1_raw, detected_raw),
    ("Top-K Voting", wf1_topk, mf1_topk, detected_topk),
    ("Neighborhood Consensus", wf1_neighbor, mf1_neighbor, detected_neighbor),
    ("Dynamic Thresholds", wf1_dynamic, mf1_dynamic, detected_dynamic),
    ("Ensemble Voting", wf1_ensemble, mf1_ensemble, detected_ensemble),
    ("⭐ ULTIMATE", wf1_ultimate, mf1_ultimate, detected_ultimate),
]

print(f"\n{'Strategy':<28} {'Weighted F1':>12} {'Macro F1':>12} {'Rooms':>10}")
print("-" * 70)
for name, wf1, mf1, rooms in results:
    marker = "🏆" if mf1 == max(r[2] for r in results) else "  "
    print(f"{marker} {name:<26} {wf1:>12.4f} {mf1:>12.4f} {rooms:>4}/{len(id2room):<3}")

# ===== PER-CLASS BREAKDOWN =====
print("\n" + "="*70)
print("📋 PER-CLASS: ULTIMATE vs BASELINE")
print("="*70)

test_classes = sorted(np.unique(yte_sorted))
test_names = [id2room[i] for i in test_classes]

report_ultimate = classification_report(yte_sorted, pred_ultimate, labels=test_classes,
                                        target_names=test_names, output_dict=True, zero_division=0)
report_baseline = classification_report(yte_sorted, pred_raw_sorted, labels=test_classes,
                                        target_names=test_names, output_dict=True, zero_division=0)

items = [(name, report_ultimate[name]) for name in test_names]
items.sort(key=lambda x: x[1]['support'], reverse=True)

print(f"{'Room':<20s} {'F1 (Base)':>10s} {'F1 (Ult)':>10s} {'Δ F1':>10s} {'Support':>10s}")
print("-" * 70)

for name, metrics_ult in items:
    metrics_base = report_baseline[name]
    delta = metrics_ult['f1-score'] - metrics_base['f1-score']

    marker = "🔺" if delta > 0.1 else "⭐" if delta > 0.05 else "🔻" if delta < -0.05 else "  "

    print(f"{marker} {name:<18s} {metrics_base['f1-score']:>10.3f} {metrics_ult['f1-score']:>10.3f} "
          f"{delta:>+10.3f} {int(metrics_ult['support']):>10d}")

print("\n✅ ULTIMATE SOLUTION: Maximum macro F1 with all classes detected!")


Majority classes (3): ['cafeteria', 'kitchen', 'nurse station']
Minority classes (19): ['501', '502', '503', '505', '506', '508', '510', '511', '512', '513', '515', '516', '517', '518', '520', '522', '523', 'cleaning', 'hallway']

STRATEGY 1: TOP-K VOTING (Consider top 3 predictions)
Weighted F1: 0.6969
Macro F1:    0.4929
Rooms detected: 21/22

STRATEGY 2: NEIGHBORHOOD CONSENSUS (±5 window voting)
Weighted F1: 0.7381
Macro F1:    0.5434
Rooms detected: 20/22

STRATEGY 3: CLASS-SPECIFIC DYNAMIC THRESHOLDS
Weighted F1: 0.7329
Macro F1:    0.5474
Rooms detected: 21/22

STRATEGY 4: ENSEMBLE VOTING (Combine all strategies)
Weighted F1: 0.7349
Macro F1:    0.5478
Rooms detected: 21/22

STRATEGY 5: ULTIMATE WEIGHTED ENSEMBLE + MINORITY BOOST

⚠️ Classes with 0% recall: ['503', '517', '518']
   Applying minority boost...
      Recovered 503 at position 2
      Recovered 503 at position 3
      Recovered 503 at position 2
      Recovered 517 at position 5

Weighted F1: 0.7403
Macro F1:    0.63

#Save Model (+Loadl later)

In [24]:
# ========================================
# Cell 11.5 — Save Trained Model + Metadata
# ========================================

import pickle
from pathlib import Path

# Define save path (UPDATE THIS)
MODEL_DIR = Path("/content/drive/MyDrive/Contest/ABC/Dataset/Models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save the trained model
model_path = MODEL_DIR / "garcia_rf_model(final).pkl"
with open(model_path, 'wb') as f:
    pickle.dump(model, f)
print(f"✓ Model saved: {model_path}")

# Save metadata (critical for inference)
metadata = {
    'room2id': room2id,
    'id2room': id2room,
    'features': FEATURES_ALL,
    'beacon_cols': BEACON_COLS,
    'mac_list': MAC_LIST,
    'beacon_id': BEACON_ID,
    'room_to_six': ROOM_TO_SIX,
    'n_classes': len(id2room),
    'window_seconds': WINDOW_SECONDS,
    'train_samples': len(train_aug),
    'test_samples': len(test_df),
    'weighted_f1': wf1_raw if 'wf1_raw' in locals() else None,
    'macro_f1': mf1_raw if 'mf1_raw' in locals() else None,
}

metadata_path = MODEL_DIR / "garcia_rf_metadata(final).pkl"
with open(metadata_path, 'wb') as f:
    pickle.dump(metadata, f)
print(f"✓ Metadata saved: {metadata_path}")

# Also save transition matrix if using hybrid smoothing
if 'trans_matrix' in locals():
    trans_matrix_path = MODEL_DIR / "transition_matrix(final).pkl"
    with open(trans_matrix_path, 'wb') as f:
        pickle.dump(trans_matrix, f)
    print(f"✓ Transition matrix saved: {trans_matrix_path}")

# Save train/test split indices for reproducibility
split_info = {
    'train_indices': train_df.index.tolist(),
    'test_indices': test_df.index.tolist(),
    'random_seed': SEED,
}
split_path = MODEL_DIR / "train_test_split(final).pkl"
with open(split_path, 'wb') as f:
    pickle.dump(split_info, f)
print(f"✓ Split info saved: {split_path}")

print("\n" + "="*70)
print("MODEL PACKAGE SAVED")
print("="*70)
print(f"Location: {MODEL_DIR}")
print(f"Files:")
print(f"  - garcia_rf_model.pkl        (RandomForest, {model.n_estimators} trees)")
print(f"  - garcia_rf_metadata.pkl     ({len(room2id)} rooms, {len(FEATURES_ALL)} features)")
print(f"  - transition_matrix.pkl      ({len(id2room)}x{len(id2room)} transitions)")
print(f"  - train_test_split.pkl       (train={len(train_df)}, test={len(test_df)})")
print("\n✅ Ready for deployment!")


✓ Model saved: /content/drive/MyDrive/Contest/ABC/Dataset/Models/garcia_rf_model(final).pkl
✓ Metadata saved: /content/drive/MyDrive/Contest/ABC/Dataset/Models/garcia_rf_metadata(final).pkl
✓ Transition matrix saved: /content/drive/MyDrive/Contest/ABC/Dataset/Models/transition_matrix(final).pkl
✓ Split info saved: /content/drive/MyDrive/Contest/ABC/Dataset/Models/train_test_split(final).pkl

MODEL PACKAGE SAVED
Location: /content/drive/MyDrive/Contest/ABC/Dataset/Models
Files:
  - garcia_rf_model.pkl        (RandomForest, 500 trees)
  - garcia_rf_metadata.pkl     (22 rooms, 33 features)
  - transition_matrix.pkl      (22x22 transitions)
  - train_test_split.pkl       (train=1631, test=700)

✅ Ready for deployment!


# # LOAD SAVED MODEL

In [ ]:


# import pickle
# from pathlib import Path

# MODEL_DIR = Path("/content/drive/MyDrive/ABC2026/Models")

# # Load model
# with open(MODEL_DIR / "garcia_rf_model.pkl", 'rb') as f:
#     model = pickle.load(f)

# # Load metadata
# with open(MODEL_DIR / "garcia_rf_metadata.pkl", 'rb') as f:
#     metadata = pickle.load(f)

# room2id = metadata['room2id']
# id2room = metadata['id2room']
# FEATURES_ALL = metadata['features']
# BEACON_COLS = metadata['beacon_cols']
# MAC_LIST = metadata['mac_list']
# BEACON_ID = metadata['beacon_id']
# ROOM_TO_SIX = metadata['room_to_six']

# # Load transition matrix (if using hybrid smoothing)
# with open(MODEL_DIR / "transition_matrix.pkl", 'rb') as f:
#     trans_matrix = pickle.load(f)

# print("✓ Model loaded successfully!")
# print(f"  Rooms: {len(id2room)}")
# print(f"  Features: {len(FEATURES_ALL)}")
# print(f"  Trained on: {metadata['train_samples']} samples")
# print(f"  Test F1: {metadata['weighted_f1']:.4f}")

# # Now you can predict on new data:
# # predictions = model.predict(X_new)
# # room_names = [id2room[pred] for pred in predictions]
